<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Batch-process MDN Chl-a algorithm
In this notebook we use the remote sensing reflectance data produced by the ACOLITE RAdCor atmospheric correction as input for the Mixture Density Networks (MDN) to produce a Chl-a product. Further information about the MDN algorithm and installation instructions can be found on the [MDN Tutorials github page](https://github.com/ryan-edward-oshea/MDN_tutorials); this notebook makes extensive use of the code provided in the MDN tutorials.  

#### This notebook does the following:
* creates a list of ACOLITE *_L2W.nc output files to process
* Runs the MDN chl-a algorithm
* Saves output to netcdf

#### Requirements: 
* Make sure that you have already created some "*_L2W.nc" output files using the *Chl_Step2_ACOLITE-RAdCor.ipynb* notebook.
* Make sure that you have set up and activated an AQV_env_11_19_25 environment according to the instructions provided [here](https://github.com/ryan-edward-oshea/MDN_tutorials)

#### Settings to adjust manually:
* Manually define the lake/dam name in code cell 4 (options: ZV, RV, TW, MV, VV, CW), default is TW. 

##### Authors:
**Marie Smith**, CSIR, South Africa

Before running this notebook, make sure that you switch to the **Python (AQV_env_11_19_25)** kernel in the dropdown at the top right of this window. This will activate the pre-installed conda environment which contains all the relevant python packages to run the MDN algorithm.

In [ ]:
'Import the base python packages needed for this notebook'
import numpy                                                          as np
import os
from   pathlib                    import Path
from   urllib.request             import urlretrieve
import zipfile
import matplotlib                                                     as mpl
import matplotlib.pyplot                                              as plt
from   matplotlib.colors          import LogNorm
from datetime import datetime
from scipy.interpolate import griddata
import xarray as xr

'Import some MDN specific functions and packages'
from   MDN                        import get_sensor_bands, get_tile_data
from   MDN                        import download_example_imagery
from   MDN                        import display_sat_rgb, find_rgb_img
from   MDN                        import map_cube_mdn_full, overlay_rgb_mdnProducts, get_tile_geographic_info

Define some useful functions to find filenames and extract dates from filenames:

In [1]:
def find_netcdf_files(base_dir):
    matching_files = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('_L2W.nc'):
                matching_files.append(os.path.join(root, file))
    return matching_files
    
def extract_datetime_from_filename(filename):
    """
    Extracts the date and time from a filename with format:
    Prefix_Year_Month_Day_Hour_Minute_Second_Suffix.nc
    Example: S2A_MSI_2024_09_11_08_49_51_T34HBH_L2W.nc
    
    Returns:
        datetime object representing the extracted date and time.
    """
    parts = filename.split('_')
    year, month, day = parts[2], parts[3], parts[4]
    hour, minute, second = parts[5], parts[6], parts[7]
    dt_str = f"{year}-{month}-{day} {hour}:{minute}:{second}"
    return datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S")

In [ ]:
## supply the name of the lake (options: ZV, RV, TW, MV, VV, CW)
lake_name = "TW"    

## create the folder where the outputs will be saved:
if not os.path.exists(lake_name+'_mdn'):
    os.makedirs(lake_name+'_mdn')
    print(f"Created folder: {lake_name}_mdn")
else:
    print(f"Folder {lake_name}_mdn already exists.") [web:1][web:5]

## Define output and input directories:
base_directory = "/"+lake_name+"_acolite/"      
output_directory = "/"+lake_name+"_mdn/"      
file_list = find_netcdf_files(base_directory)

# Create list of (filepath, time) tuples
files_with_time = [(f, extract_datetime_from_filename(os.path.basename(f))) for f in file_list]

# Sort by extracted time
files_with_time_sorted = sorted(files_with_time, key=lambda x: x[1])

# Separate sorted filepaths and time
sorted_file_list = [f[0] for f in files_with_time_sorted]
sorted_times = [f[1] for f in files_with_time_sorted]

print(len(sorted_file_list))

Below are all the options for the SWAM lakes which can be used to define the target grid for MDN in the next code cell

In [ ]:
"""
# RV
# POLYGON((18.476077 -33.856497, 18.476077 -33.825699, 18.519984 -33.825699, 18.519984 -33.856497, 18.476077 -33.856497))
lon_min = 18.476077
lon_max = 18.519984
lat_min = -33.856497
lat_max = -33.825699

# CW
# POLYGON((18.862152 -32.277845, 18.862152 -32.181424, 18.939056 -32.181424, 18.939056 -32.277845, 18.862152 -32.277845))
lon_min = 18.862152
lon_max = 18.939056
lat_min = -32.277845
lat_max = -32.181424

# MV
# POLYGON((18.786278 -33.074224, 18.786278 -33.022136, 18.832626 -33.022136, 18.832626 -33.074224, 18.786278 -33.074224))
lon_min = 18.786278
lon_max = 18.832626
lat_min = -33.074224
lat_max = -33.022136

# VV
# POLYGON((19.018662 -33.406898, 19.018662 -33.33235, 19.066678 -33.33235, 19.066678 -33.406898, 19.018662 -33.406898))
lon_min = 19.018662
lon_max = 19.066678
lat_min = -33.406898
lat_max = -33.33235

# TW  
# POLYGON((19.120331 -34.081668, 19.120331 -33.983225, 19.293365 -33.983225, 19.293365 -34.081668, 19.120331 -34.081668))
lon_min = 19.120331
lon_max = 19.293365
lat_min = -34.081668
lat_max = -33.983225

# ZK  Zeekoevlei
lon_min = 18.451901
lon_max = 18.532856
lat_min = -34.100973
lat_max = -34.042389
"""

In the next step we are defining the target grid for the data regridding step. 

In [ ]:
# Define target grid (per the original SWAM project subsetting)
lon_min = 19.120331
lon_max = 19.293365
lat_min = -34.081668
lat_max = -33.983225

# Define the spatial resoltion (default 20m as provided below)
lat_res = 0.00018
lon_res = 0.00022

num_lon = int(round((lon_max - lon_min) / lon_res)) + 1
num_lat = int(round((lat_max - lat_min) / lat_res)) + 1

target_lat = np.linspace(lat_min, lat_max, num_lat)
target_lon = np.linspace(lon_min, lon_max, num_lon)

# Create meshgrid for target points
target_lon_grid, target_lat_grid = np.meshgrid(target_lon, target_lat)
target_points = np.array([target_lat_grid.flatten(), target_lon_grid.flatten()]).T

Below is the main processing step where you read the input file location, apply the MDN algorithm, regrid the data, and write the netcdf output files. This can take a few minutes to complete, so be patient! 


In [ ]:
sensor= "MSI"
#products = "chl,tss,cdom"
products = "chl"

for file in sorted_file_list:
    satellite = os.path.basename(file)[0:4]
    datestr = os.path.basename(file)[8:18]

    #Let us now load the Rrs data
    bands, Rrs =  get_tile_data(file, sensor)
    if np.all(np.isnan(Rrs)):
        nan_array = np.full_like(target_lat_grid, np.nan)
        da_regridded = xr.DataArray(
            nan_array,
            dims=['lat', 'lon'],
            coords={'lat': target_lat, 'lon': target_lon},
            name='chl'
        )
        # Save or further process regridded data
        savename = os.path.basename(file)[:-6] + 'chl'
        da_regridded.to_netcdf(output_directory + savename + '.nc')
        del nan_array, da_regridded, savename
        continue  # Skip to the next file in the loop
    
    #Get the geographic information
    from MDN.utils import get_tile_geographic_info
    lon, lat, extent = get_tile_geographic_info(file)

    #Run MDNs on Rrs to create a product object for image mapping
    from MDN.utilities import map_cube_mdn_full
    from MDN.parameters import get_args
    kwargs= {'sensor':sensor,'products':products,'model_loc': "Weights" ,'sat_bands': False, 'min_in_out_val':1e-6,  'model_uid':"1b79098defa9693eb03ac734467c3854cf4fb81ce3d1a96b9bf0f511f1c997b1"}
    
    try:
        args = get_args(kwargs, use_cmdline=False)
        print(f"Processing {os.path.basename(file)}")
        
        model_preds, (model_uncert), slices = map_cube_mdn_full(
            args, Rrs, bands, land_mask=False, flg_subsmpl=False,
            scaler_mode="invert", block_size=10000, uncert_mode="composite"
        )
        
        chl_preds = np.squeeze(model_preds[:,:, slices['chl']])
        chl_uncerts = np.squeeze(model_uncert[:,:, slices['chl']])
        
        # Clean up the data
        test = np.where(chl_preds == 0, np.nan, chl_preds)
        test2 = np.where(test > 1000, 1000, test)
        data = np.where(test2 < 0.01, 0.01, test2)
        
    except ValueError as e:
        print(f"ValueError in {os.path.basename(file)}: {str(e)} - Skipping to NaN output")
        # Create NaN array for failed MDN processing
        data = np.full_like(Rrs[:,:,0], np.nan)  # Match expected shape
        
    except Exception as e:
        print(f"Unexpected error in {os.path.basename(file)}: {str(e)} - Skipping to NaN output")
        data = np.full_like(Rrs[:,:,0], np.nan)

    lat_array = lat   # shape (ny, nx)
    lon_array = lon  # shape (ny, nx)
    
    # Extract 1D coordinate arrays assuming these are meshgrids
    lat_1d = lat_array[:, 0]
    lon_1d = lon_array[0, :]
    
    # Flatten the coordinate meshgrid and data for input points
    points = np.array([lat_array.flatten(), lon_array.flatten()]).T
    values = data.flatten()
    
    # Interpolate using linear method (alternatives: 'nearest', 'cubic')
    regridded_values = griddata(
        points, values, target_points, method='linear', fill_value=np.nan
    )
    
    # Reshape interpolated values to 2D grid
    regridded_data = regridded_values.reshape(target_lat_grid.shape)
    
    # Create new xarray DataArray with regridded data
    da_regridded = xr.DataArray(
        regridded_data,
        dims=['lat', 'lon'],
        coords={'lat': target_lat, 'lon': target_lon},
        name='chl'
    )
    
    # Save or further process regridded data
    savename = os.path.basename(file)[:-6] + 'chl'
    da_regridded.to_netcdf(output_directory + savename + '.nc')
    print('saved '+savename)
    
    # Clear the variable from memory
    del bands, Rrs, lon, lat, extent
    if 'model_preds' in locals():
        del model_preds, model_uncert, slices, chl_preds, chl_uncerts, test, test2, data, regridded_values, regridded_data, da_regridded